<a href="https://colab.research.google.com/github/c-marq/cap4767-data-mining/blob/main/exercises/m06-07_group_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modules 6–7 Group Exercise — RFM + K-Means on a New Dataset
**CAP4767 Data Mining with Python** | Miami Dade College — Kendall Campus

**Points:** 10 | **Duration:** ~45 minutes | **Deliverable:** Completed notebook + 3-minute presentation

**Objective:** Run the complete RFM + K-Means pipeline on a **different dataset** from the demo. Apply what you learned in both sessions — RFM scoring, CLTV, manual segmentation, K-Means clustering, and segment comparison.

**Two sessions:** Session 1 is Tasks 1 and 2 (RFM metrics, scores and manual segments), after the Chapter 6 demo. Session 2 is Tasks 3 to 5 (scaling, elbow, K-Means and comparison), after the Chapter 7 demo, then the share-outs.

**Companion page:** the Modules 6–7 practice worksheet walks through these same steps on the same two datasets, with hints and solutions. https://c-marq.github.io/cap4767-data-mining/practice/m06-07-segmentation/

### Group Members & Roles

| Role | Name | Responsibility |
|------|------|----------------|
| 🖥️ **Lead Coder** | | Drives the notebook |
| 📊 **Data Interpreter** | | Reads outputs, explains what the numbers mean |
| 🎤 **Presenter** | | Delivers the 3-minute share-out |
| ✅ **QA Reviewer** | | Checks outputs against checkpoints |

*If 3 members, Lead Coder also handles QA.*

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 GROUP DISCUSSION (before coding — 3 minutes)</strong><br>
  Your dataset tracks a different type of customer than the UK online retailer from the demo. Before running any code:
  <ol>
    <li>What does <strong>Recency</strong> mean in this context? What column represents it, or how will you calculate it?</li>
    <li>What does <strong>Frequency</strong> mean? Is it the number of orders, visits, or something else?</li>
    <li>What does <strong>Monetary</strong> mean? Is it revenue, volume, or a different measure?</li>
    <li>Do you expect the Pareto concentration to be more or less extreme than the Online Retail dataset? Why?</li>
  </ol>
</div>

**Our group's answers (2–3 sentences per question):**

*(Write here)*

---

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Run the setup cell. Then choose ONE dataset in the data loading cell: it is set to Option B (the restaurant data). To use Option A (Olist) instead, change the first line to <code>DATASET = "A"</code>. Run the cell.
</div>

In [ ]:
# ============================================================
# Setup — Run this cell. Do not modify.
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

# Segmentation function from the demo
def segment_customer(score):
    if score >= 12:
        return "Champions"
    elif score >= 9:
        return "Loyal"
    elif score >= 7:
        return "Potential Loyalist"
    elif score >= 5:
        return "At Risk"
    elif score >= 4:
        return "Can't Lose"
    else:
        return "Lost"

print("✅ Setup complete — segment_customer() function loaded")

In [ ]:
# ============================================================
# CHOOSE YOUR DATASET: change "B" to "A" on the next line to use Option A
# ============================================================
DATASET = "B"   # "A" = Brazilian E-Commerce (Olist), more challenging: merges 3 tables | "B" = Restaurant Orders, simpler

if DATASET == "A":
    # OPTION A: Brazilian E-Commerce (Olist), about 100K orders
    base = "https://raw.githubusercontent.com/c-marq/cap4767-data-mining/refs/heads/main/data"
    customers = pd.read_csv(f"{base}/olist_customers_dataset.csv")
    orders = pd.read_csv(f"{base}/olist_orders_dataset.csv")
    payments = pd.read_csv(f"{base}/olist_order_payments_dataset.csv")

    # Merge: orders → customers (to get customer_unique_id) → payments (to get revenue)
    merged = orders.merge(customers, on="customer_id").merge(payments, on="order_id")

    # Use customer_unique_id (not customer_id — one person can have multiple customer_ids)
    df = merged[["customer_unique_id", "order_purchase_timestamp", "order_id", "payment_value"]].copy()
    df.columns = ["CustomerID", "InvoiceDate", "InvoiceNo", "TotalPrice"]
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df = df.dropna(subset=["CustomerID"])
    df = df[df["TotalPrice"] > 0]
    DOMAIN = "Brazilian E-Commerce"
else:
    # OPTION B: Restaurant Orders, about 13,400 orders (74,818 order lines)
    url = "https://raw.githubusercontent.com/c-marq/cap4767-data-mining/refs/heads/main/data/restaurant-1-orders.csv"
    df_raw = pd.read_csv(url)

    # Restaurant data doesn't have CustomerID — use Order Number as proxy
    # Each order is one "visit" — Frequency = number of visits
    df = df_raw.copy()
    df["TotalPrice"] = df["Quantity"] * df["Product Price"]
    # The dates are day first (31/07/2019 means 31 July), so we give pandas the format
    df["InvoiceDate"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y %H:%M")
    df.rename(columns={"Order Number": "InvoiceNo"}, inplace=True)

    # Create a CustomerID proxy: since we don't have one, group by Order Number
    # For this dataset, each order IS a customer visit — we'll aggregate at order level first
    # then treat each unique order as a "customer" for RFM
    # NOTE: This is a simplification — discuss with your group why this is a limitation
    order_summary = df.groupby("InvoiceNo").agg(
        InvoiceDate=("InvoiceDate", "first"),
        TotalPrice=("TotalPrice", "sum")
    ).reset_index()
    order_summary["CustomerID"] = order_summary["InvoiceNo"]  # Each order = unique customer
    df = order_summary
    DOMAIN = "Restaurant Orders"

print(f"Dataset: {DOMAIN}")
print(f"Shape: {df.shape[0]:,} rows")
print(f"Unique customers/orders: {df['CustomerID'].nunique():,}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
df.head()

<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #7D6608;">⚠️ DATASET NOTES</strong><br>
  <ul>
    <li><strong>Option A (Olist):</strong> Uses <code>customer_unique_id</code> (not <code>customer_id</code>) because one person can have multiple IDs across orders. Revenue = <code>payment_value</code>. Check your load: the last order is on 2018-10-17.</li>
    <li><strong>Option B (Restaurant):</strong> No true CustomerID — each order number is treated as a unique customer. This limits RFM interpretation (Frequency will always be 1). Discuss this limitation in your share-out. Check your load: the date range should run from 2015-09-01 to 2019-08-03.</li>
    <li><strong>Frequency ties (both options):</strong> <code>.rank(method='first')</code> breaks ties by row order. When almost everyone has Frequency 1 (all of Option B, about 97% of Option A), F_Score is just row order. In Option B it almost copies R_Score. Say what that means for your clusters in your share-out.</li>
  </ul>
</div>

---
## Task 1 — Calculate RFM Metrics (1 code cell)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Set a snapshot date (1 day after the last transaction). Use <code>groupby</code> on CustomerID to calculate Recency, Frequency, and Monetary, and store the table as <code>rfm</code> with the columns <code>CustomerID</code>, <code>Recency</code>, <code>Frequency</code> and <code>Monetary</code>. Print <code>rfm.describe()</code>.
</div>

In [ ]:
# Task 1: Calculate RFM metrics
# YOUR CODE HERE


<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 1</strong><br>
  Your <code>rfm</code> DataFrame should have one row per customer and three numeric columns. Recency should be in days, Frequency should use <code>nunique</code>, Monetary should be a sum. <strong>Option B:</strong> 13,397 rows, and Frequency is 1 for every row. <strong>Option A:</strong> about 96,000 rows (anything from 90,000 to 98,000 is fine) and a largest Frequency between 5 and 20 (we get 17). If you have about 99,000 rows, you grouped by <code>customer_id</code> instead of <code>customer_unique_id</code>. If your largest Frequency is over 20, you counted rows instead of unique orders.
</div>

---
## Task 2 — Assign RFM Scores and Manual Segments (1 code cell)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Apply <code>pd.qcut()</code> for R, F, M quintile scores (1–5) and store them as <code>R_Score</code>, <code>F_Score</code> and <code>M_Score</code>, each ending in <code>.astype(int)</code> (without it the scores cannot be added or averaged). Use <code>.rank(method='first')</code> on Frequency if you get a bin-edge error. Calculate the composite score as <code>RFM_Score</code>. Apply <code>segment_customer()</code> and store the labels in <code>rfm["Customer_Segment"]</code>. Print value_counts.
</div>

In [ ]:
# Task 2: RFM scores + manual segments
# YOUR CODE HERE


**Interpretation:** Which segment is largest? Which has the fewest customers? Does this distribution surprise you?

*(Write here)*

<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 2</strong><br>
  You should have 6 segment labels, and your Champions should be recent buyers: the median Recency of the Champions must be below the median Recency of all customers (we get about 128 against 474 days in Option B, and about 150 against 269 in Option A). If it is higher, the Recency labels were not reversed: check <code>labels=[5,4,3,2,1]</code>. In Option B a missed reversal shows only 3 labels and no Champions at all.
</div>

---
## Task 3 — Scale Features and Run Elbow Method (1 code cell)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Use <code>StandardScaler</code> on R, F, M scores and store the result as <code>features_scaled</code> (Task 4 uses that name). Run K-Means for k=2 through k=8 with <code>random_state=42, n_init=10</code>. Plot the inertia curve. Start your inertia list empty <em>inside</em> this cell, so running the cell twice does not add to the old list.
</div>

In [ ]:
# Task 3: Scale + Elbow Method
# YOUR CODE HERE


**Chosen k and justification (1–2 sentences):**

*(Write here)*

<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 3</strong><br>
  Print <code>features_scaled.std(axis=0).round(2)</code>: every column should show 1.0 (a column that is the same for everyone shows 0.0, which is fine). A value like 1.41, or hundreds, means the scaler was skipped. Your inertia should fall at every step, with smaller drops as k grows. There may be no sharp bend (ours had none), so choose a k and say why. An error about "same first dimension" means your inertia list is longer than your list of k values.
</div>

---
## Task 4 — Fit K-Means and Profile Clusters (1 code cell)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Fit K-Means with your chosen k (use <code>random_state=42, n_init=10</code>). Add the cluster labels to <code>rfm</code> as <code>rfm["Cluster"]</code> (Task 5 uses that name). Create a cluster profiling heatmap (mean R, F, M per cluster). Print cluster sizes.
</div>

In [ ]:
# Task 4: K-Means + profiling heatmap
# YOUR CODE HERE


**Name each cluster (1 sentence per cluster):**

*(Write here)*

---
## Task 5 — Compare Manual Segments vs K-Means (1 code cell)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Build a crosstab: <code>pd.crosstab(rfm["Customer_Segment"], rfm["Cluster"], margins=True)</code>
</div>

In [ ]:
# Task 5: Crosstab comparison
# YOUR CODE HERE


**Interpretation (2–3 sentences):** Where do the two approaches agree? Where do they disagree? Did K-Means discover a sub-group within one of the manual segments?

*(Write here)*

---
## Six Success Criteria Questions

Answer each in 1–3 sentences:

1. **How many unique customers are in your dataset after cleaning?**
2. **What percentage of customers fall into the top two RFM segments (Champions + Loyal)?**
3. **What is the ratio of mean to median Monetary value?** (Revenue concentration indicator)
4. **How many clusters did the Elbow Method suggest, and do you agree?**
5. **Which K-Means cluster represents the highest-value customers, and how many are in it?**
6. **Did K-Means reveal a group that the manual rules missed or split?** Describe it.

*(Write answers here)*

---

## Share-Out (3 minutes)

Present to the class:
1. Which dataset your group used and how you mapped R, F, M
2. Your most interesting finding from manual segmentation
3. Your most interesting finding from K-Means
4. One recommendation a business could act on based on your segmentation

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `pd.qcut` error: "Bin edges must be unique" | Add `.rank(method='first')` before qcut |
| Champions look old, or Option B shows only 3 segments | Check Recency label reversal: `labels=[5,4,3,2,1]` |
| Olist merge raises `KeyError: 'CustomerID'` | Merge on `customer_id` and `order_id` (the original names); rename to `CustomerID` after the merge |
| `TypeError: unsupported operand type(s) for +: 'Categorical' and 'Categorical'` (or `category dtype does not support aggregation 'mean'`) | Add `.astype(int)` to each `pd.qcut()` line |
| `KeyError: 'Customer_Segment'` or `KeyError: 'Cluster'` in Task 5 | Store the labels as `rfm["Customer_Segment"]` (Task 2) and `rfm["Cluster"]` (Task 4) |
| Elbow cell: `ValueError: x and y must have same first dimension` | Start the inertia list empty inside the cell; re-running it added to the old list |
| Restaurant Frequency is all 1 | Expected: each order is a unique "customer." F_Score is then only row order and almost copies R_Score. Discuss it as a limitation. |

---
<p style="color:#7F8C8D; font-size:0.85em;">
<em>CAP4767 Data Mining with Python | Miami Dade College</em><br>
Modules 6–7 Group Exercise — RFM + K-Means on a New Dataset | 10 Points
</p>